# E0006 — CUDA 12.9 offline wheelhouse builder (NO GPU)

Purpose: download the exact vLLM 0.27.1 + CUDA 12.9 runtime closure for later **offline** Kaggle use, without spending L4 quota.

Required settings:
- Accelerator: **None**
- Internet: **ON**

This notebook:
1. downloads the exact CUDA 12.9 wheel set into `/kaggle/working/e0006_cu129_wheelhouse`;
2. explicitly includes `flashinfer-cubin==0.6.16.post3` for offline kernels on supported GPUs such as L4/SM89;
3. validates dependency closure with a **no-index dry run** from the downloaded wheelhouse only;
4. writes a SHA-256 manifest.

It does **not** install the runtime, load Nemotron, or use GPU.

Expected outputs:
- `/kaggle/working/e0006_cu129_wheelhouse/`
- `/kaggle/working/e0006_cu129_wheelhouse_manifest.json`


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

OUT_DIR = Path("/kaggle/working/e0006_cu129_wheelhouse")
MANIFEST = Path("/kaggle/working/e0006_cu129_wheelhouse_manifest.json")
OFFLINE_REPORT = Path("/kaggle/working/e0006_cu129_offline_resolution_report.json")

VLLM_WHEEL = (
    "https://github.com/vllm-project/vllm/releases/download/v0.27.1/"
    "vllm-0.27.1%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl"
)
VLLM_SHA256 = "bf0d52faa2a51e7a01c6856a7a8a2d1307fd0ff711415d34168a67ffac0fa47b"
FLASHINFER_CUBIN = "flashinfer-cubin==0.6.16.post3"

free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Free /kaggle/working space: {free_gib:.2f} GiB")
if free_gib < 9.0:
    raise RuntimeError(
        f"STOP: only {free_gib:.2f} GiB free; require >=9 GiB before building the wheelhouse."
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)

download_cmd = [
    sys.executable, "-m", "pip", "download",
    "--only-binary=:all:",
    "--dest", str(OUT_DIR),
    "--extra-index-url", "https://download.pytorch.org/whl/cu129",
    "--extra-index-url", "https://flashinfer.ai/whl/",
    VLLM_WHEEL,
    FLASHINFER_CUBIN,
]
print("Downloading exact CUDA 12.9 wheelhouse...")
cp = subprocess.run(download_cmd, text=True)
if cp.returncode != 0:
    raise RuntimeError(f"pip download failed with return code {cp.returncode}")

files = sorted(p for p in OUT_DIR.iterdir() if p.is_file())
if not files:
    raise RuntimeError("Wheelhouse is empty after pip download.")

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

entries = []
for p in files:
    entries.append({
        "name": p.name,
        "bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

vllm_entries = [e for e in entries if e["name"].startswith("vllm-0.27.1+cu129")]
if len(vllm_entries) != 1:
    raise RuntimeError(f"Expected exactly one vLLM +cu129 wheel, found {len(vllm_entries)}")
if vllm_entries[0]["sha256"] != VLLM_SHA256:
    raise RuntimeError(
        "Frozen vLLM wheel SHA-256 mismatch: "
        f"{vllm_entries[0]['sha256']} != {VLLM_SHA256}"
    )

cubin_entries = [e for e in entries if e["name"].startswith("flashinfer_cubin-0.6.16.post3")]
if len(cubin_entries) != 1:
    raise RuntimeError(
        f"Expected exactly one flashinfer-cubin 0.6.16.post3 wheel, found {len(cubin_entries)}"
    )

local_vllm = OUT_DIR / vllm_entries[0]["name"]
offline_cmd = [
    sys.executable, "-m", "pip", "install",
    "--dry-run",
    "--ignore-installed",
    "--no-index",
    "--find-links", str(OUT_DIR),
    "--report", str(OFFLINE_REPORT),
    str(local_vllm),
    FLASHINFER_CUBIN,
]
print("Validating dependency closure from wheelhouse only (no index)...")
offline = subprocess.run(offline_cmd, capture_output=True, text=True, timeout=900)

total_bytes = sum(e["bytes"] for e in entries)
payload = {
    "experiment": "E0006",
    "gate": "D2_CU129_WHEELHOUSE_BUILD_NO_GPU",
    "status": "WHEELHOUSE_READY" if offline.returncode == 0 else "WHEELHOUSE_INCOMPLETE",
    "python": sys.version,
    "download_command": download_cmd,
    "offline_validation_command": offline_cmd,
    "offline_validation_returncode": offline.returncode,
    "offline_stdout_tail": offline.stdout[-12000:],
    "offline_stderr_tail": offline.stderr[-12000:],
    "file_count": len(entries),
    "total_bytes": total_bytes,
    "total_gib": round(total_bytes / 1024**3, 3),
    "frozen_vllm_sha256": VLLM_SHA256,
    "flashinfer_cubin_present": bool(cubin_entries),
    "files": entries,
    "next_rule": (
        "Do not spend L4 yet. Attach this saved notebook output to a no-GPU offline import-smoke notebook first."
    ),
}
MANIFEST.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print(json.dumps({
    "status": payload["status"],
    "file_count": payload["file_count"],
    "total_gib": payload["total_gib"],
    "offline_validation_returncode": offline.returncode,
    "flashinfer_cubin_present": payload["flashinfer_cubin_present"],
}, indent=2))
print(f"WROTE: {MANIFEST}")

if offline.returncode != 0:
    raise RuntimeError(
        "Offline wheelhouse closure failed. Inspect manifest and offline report; do not proceed."
    )
